# SnapFit clothes parser fine-tuning (SegFormer-B2, Western + South Asian wear)

This notebook teaches SnapFit's clothes parser (`mattmdjaga/segformer_b2_clothes`)
to label a **kameez / kurta as one top** (label 4 Upper-clothes), a **shalwar as
pants** (6) and a **dupatta as a scarf** (17), without forgetting Western clothes.
Nobody has to draw labels by hand:

1. **Specs** – a balanced list of ~1200 outfits (kameez, kurta, and Western shirts
   and t-shirts as controls).
2. **Generate** – Stable Diffusion 1.5 draws one full-body photo per spec, on both GPUs.
3. **Auto-label** – the current parser labels each photo; its known mistakes are fixed
   using the spec (we know what the person wears); unclear photos are dropped.
4. **Western data** – the ATR clothes dataset, with a fixed held-out test set.
5. **Train** – fine-tune on a 50/50 mix (synthetic + Western).
6. **Evaluate** – base vs fine-tuned on Western photos and on held-out synthetic photos.
7. **Gate + save** – the model is marked *recommended* only if Western quality stays
   within 1 mIoU point and the kameez result improves.

**How to run:** Session options → Accelerator **GPU T4 x2**, Internet **On**. Then
**Save Version → Save & Run All (Commit)**. It takes about 3–4 hours. Every stage
prints its progress. Results: `/kaggle/working/report/` and
`/kaggle/working/segformer-snapfit/`. Guide: `snapfit-ai/gpu/finetune/README.md`.

## Settings
The numbers at the top control the run time. The defaults fit a T4 x2 session in
about 3–4 hours (Kaggle's limit is 12 hours).

In [ ]:
import glob
import json
import math
import os
import random
import shutil
import subprocess
import sys
import tarfile
import time
import traceback

RUN_START = time.time()

# ---- How much work (main time controls) ----
NUM_IMAGES = 1200             # synthetic photos to generate (both GPUs share them)
GEN_TIME_BUDGET_MIN = 80      # generation stops after this many minutes, however many are done
GEN_STEPS = 22                # Stable Diffusion steps per photo (DPM++ 2M scheduler)
GEN_GUIDANCE = 7.0
GEN_WIDTH, GEN_HEIGHT = 512, 768   # portrait
GEN_BATCH = 2                 # photos per GPU call
TRAIN_TIME_BUDGET_MIN = 75    # training stops after this many minutes (or MAX_EPOCHS)
MAX_EPOCHS = 15
TOTAL_TIME_BUDGET_H = 5.0     # whole notebook; training is shortened if earlier stages ran long

# ---- Data ----
SPEC_SEED = 1234
USE_SAM = True                # optional SAM edge refinement (any SAM error = skipped)
WESTERN_TEST_SIZE = 500       # fixed held-out ATR test set (never trained on)
WESTERN_VAL_SIZE = 300        # ATR set used to pick the best epoch
WESTERN_SPLIT_SEED = 2026
MIN_WESTERN_TRAIN, MAX_WESTERN_TRAIN = 500, 3000
MIN_SYNTH_TRAIN = 50          # fewer kept synthetic photos than this = no training

# ---- Training ----
BATCH_PER_GPU = 8             # halved automatically if it does not fit
LEARNING_RATE = 6e-5
WEIGHT_DECAY = 0.01
WARMUP_STEPS = 100
POLY_POWER = 0.9

# ---- Gate ----
MAX_WESTERN_DROP = 1.0        # Western mIoU may drop by at most 1.0 point

# ---- End of run ----
CLEANUP_AT_END = True         # pack the synthetic data into one archive, delete the work folder
KEEP_SYNTH_ARCHIVE = True     # keep that archive in the output (~1 GB)

# ---- Models and data (all openly licensed) ----
SD_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"   # CreativeML OpenRAIL-M
BASE_PARSER = "mattmdjaga/segformer_b2_clothes"            # MIT
SAM_MODEL = "facebook/sam-vit-base"                        # Apache-2.0
WESTERN_DATASET = "mattmdjaga/human_parsing_dataset"       # ATR copy on Hugging Face

# ---- Folders ----
WORK = "/kaggle/working"
LIB = os.path.join(WORK, "snapfit_ft")            # helper modules (written below)
FT = os.path.join(WORK, "ft_work")                # intermediate files (resumable)
SYNTH = os.path.join(FT, "synth")
LOGS = os.path.join(FT, "logs")
CKPT = os.path.join(FT, "checkpoints")
MODEL_OUT = os.path.join(WORK, "segformer-snapfit")
REPORT = os.path.join(WORK, "report")
for folder in (LIB, FT, SYNTH, LOGS, CKPT, REPORT):
    os.makedirs(folder, exist_ok=True)
for sub in ("images", "specs", "flagged", "parsed", "target"):
    os.makedirs(os.path.join(SYNTH, sub), exist_ok=True)
open(os.path.join(LIB, "__init__.py"), "a").close()

TIMINGS = {}


def elapsed_min():
    return (time.time() - RUN_START) / 60.0


def stage(title):
    line = f"[{elapsed_min():6.1f} min] ===== {title} ====="
    print("\n" + line, flush=True)
    with open(os.path.join(REPORT, "progress.log"), "a", encoding="utf-8") as handle:
        handle.write(line + "\n")
    return time.time()


def minutes_left():
    return TOTAL_TIME_BUDGET_H * 60.0 - elapsed_min()


print("Settings ready. Work folder:", FT)

## Check the session (GPU, packages, internet)

In [ ]:
stage("Checking the session")
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Session options -> Accelerator -> GPU T4 x2, then run again.")
N_GPUS = torch.cuda.device_count()
for index in range(N_GPUS):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name}, {props.total_memory / 2**30:.1f} GB")
if N_GPUS < 2:
    print("NOTE: only one GPU. Everything still works, but generation takes about twice as long;"
          " it stops at GEN_TIME_BUDGET_MIN with fewer photos.")

for package in ("diffusers", "transformers", "datasets", "accelerate"):
    try:
        __import__(package)
    except ImportError:
        print(f"{package} missing, installing it ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import datasets
import diffusers
import transformers

print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__)

try:
    from huggingface_hub import model_info

    model_info(BASE_PARSER, timeout=30)
    print("Internet: OK (Hugging Face reachable)")
except Exception as error:
    raise RuntimeError("Cannot reach Hugging Face. Session options -> Internet: On, then run again. "
                       f"({error!r})")

free_gb = shutil.disk_usage(WORK).free / 2**30
print(f"Free disk in {WORK}: {free_gb:.0f} GB")

## Helper modules
The next cells save small helper files (the same files as in
`snapfit-ai/gpu/finetune/` in the SnapFit repo). They are plain Python and are
tested there with pytest.

In [ ]:
%%writefile /kaggle/working/snapfit_ft/prompts.py
"""Generation specs for the synthetic clothes-parsing set (pure Python).

Each spec describes one full-body photo to generate with Stable Diffusion 1.5:
who (gender, age), what they wear (top, bottom, dupatta, colours, pattern),
where (background) and how they stand (pose). It also carries the labels the
picture should get, in the 18 ATR ids the parser already uses, so the
auto-labelling step knows what to expect:

    kameez / kurta (whole garment, long lower part and sleeves) -> 4 Upper-clothes
    shirt / t-shirt (Western controls)                          -> 4 Upper-clothes
    shalwar, trousers, jeans                                     -> 6 Pants
    dupatta                                                      -> 17 Scarf

The list is balanced by construction (exact shares per garment group and
gender), seeded (same seed -> same list) and shuffled, so any first part of
it (for example when generation stops at its time budget) is still mixed.
"""

import random
import re

UPPER_CLOTHES = 4
PANTS = 6
SCARF = 17

# (top, gender, share of all specs). Shares add up to 1.
GROUPS = [
    ("kameez", "woman", 0.20),
    ("kameez", "man", 0.15),
    ("kurta", "woman", 0.15),
    ("kurta", "man", 0.15),
    ("shirt", "woman", 0.0875),
    ("shirt", "man", 0.0875),
    ("tshirt", "woman", 0.0875),
    ("tshirt", "man", 0.0875),
]

LONG_TOPS = ("kameez", "kurta")
# Share of women's kameez / kurta specs that get a dupatta (men never do).
DUPATTA_SHARE = {"kameez": 0.7, "kurta": 0.4}

# Lowest row of the top, as a share of the person's height (head top = 0,
# feet = 1). The hip/crotch line is near 0.5, the knee near 0.72.
# Long tops must reach at least this far down to count as long ...
MIN_HEM = {"kameez": 0.55, "kurta": 0.50}
# ... and short tops must stop above this.
MAX_HEM = {"shirt": 0.60, "tshirt": 0.60}

BOTTOMS = {
    "kameez": ["shalwar"],
    "kurta": ["shalwar", "shalwar", "trousers", "jeans"],
    "shirt": ["trousers", "jeans"],
    "tshirt": ["trousers", "jeans"],
}

COLOURS = [
    "white", "black", "navy blue", "sky blue", "maroon", "red", "pink",
    "peach", "mustard yellow", "olive green", "emerald green", "teal",
    "grey", "beige", "cream", "purple", "lavender", "brown", "orange",
    "off-white",
]
PATTERNS = {
    "kameez": ["plain", "neck-embroidered", "block printed", "floral printed", "lawn printed", "lace-trimmed"],
    "kurta": ["plain", "embroidered", "chikankari", "buttoned", "printed"],
    "shirt": ["plain", "striped", "checked", "plain oxford"],
    "tshirt": ["plain", "striped", "plain crew neck"],
}
FABRICS = {
    "kameez": ["cotton", "lawn", "linen", "khaddar", "silk"],
    "kurta": ["cotton", "linen", "khaddar", "silk"],
    "shirt": ["cotton", "linen", "denim"],
    "tshirt": ["cotton", "jersey"],
}
AGES = ["young", "adult", "middle-aged"]
WESTERN_PEOPLE = ["South Asian", "Pakistani", "European", "African", "East Asian", "Middle Eastern", "Latino"]
SOUTH_ASIAN_PEOPLE = ["Pakistani", "Pakistani", "South Asian", "Indian", "Bangladeshi"]
BACKGROUNDS = [
    "plain light grey studio backdrop",
    "plain white studio wall",
    "softly lit photo studio",
    "modern living room",
    "bedroom with a wardrobe",
    "office corridor",
    "quiet city street",
    "market street",
    "park with trees",
    "courtyard with brick walls",
    "shopping mall hallway",
    "house front with a gate",
]
POSES = [
    "standing straight, arms at the sides",
    "A-pose, arms slightly away from the body",
    "standing, arms slightly out from the hips",
    "standing relaxed, arms at the sides",
    "standing upright, hands open beside the thighs",
]

FRAMING = "front view, head to toe, feet visible, one person"
QUALITY = "realistic photo, sharp focus"
NEGATIVE_PROMPT = (
    "cropped, close-up, portrait, cut off feet, cut off head, two people, multiple people, crowd, "
    "mirror, reflection, back view, side view, sitting, nsfw, nude, deformed, extra arms, extra legs, "
    "bad anatomy, blurry, lowres, text, watermark, logo, cartoon, drawing, anime, 3d render"
)

# Every 14 specs: 2 go to the held-out test set, 1 to validation (used to
# pick the best epoch), 11 to training.
SPLIT_CYCLE = 14
TEST_POSITIONS = (3, 10)
VAL_POSITIONS = (0,)


def split_for(index):
    position = index % SPLIT_CYCLE
    if position in TEST_POSITIONS:
        return "test"
    if position in VAL_POSITIONS:
        return "val"
    return "train"


def _group_counts(total):
    """Exact number of specs per group, adding up to `total` (largest
    remainder rounding, so shares are as close as possible)."""
    raw = [share * total for _, _, share in GROUPS]
    counts = [int(value) for value in raw]
    order = sorted(range(len(GROUPS)), key=lambda i: raw[i] - counts[i], reverse=True)
    for i in order[: total - sum(counts)]:
        counts[i] += 1
    return counts


def _cycle(rng, options, count):
    """`count` picks from `options`, each used as evenly as possible, in a
    seeded random order."""
    picks = [options[i % len(options)] for i in range(count)]
    rng.shuffle(picks)
    return picks


def _article(word):
    return "an" if word[:1].lower() in "aeiou" else "a"


def _garment_phrase(top, colour, pattern, fabric):
    pattern_text = "" if pattern == "plain" else pattern + " "
    kind = {
        "kameez": "knee-length kameez with long sleeves",
        "kurta": "mid-thigh kurta with long sleeves",
        "shirt": "long-sleeve shirt",
        "tshirt": "short-sleeve t-shirt",
    }[top]
    return f"{_article(colour)} {colour} {pattern_text}{fabric} {kind}"


def _bottom_phrase(bottom, colour):
    if bottom == "shalwar":
        return f"loose {colour} shalwar"
    if bottom == "jeans":
        return f"{colour} jeans"
    return f"{colour} trousers"


def _bottom_colour(rng, bottom, top_colour):
    if bottom == "jeans":
        return rng.choice(["blue", "dark blue", "light blue", "black", "grey"])
    if bottom == "shalwar":
        # A matching suit is the usual case.
        return top_colour if rng.random() < 0.6 else rng.choice(COLOURS)
    return rng.choice(["black", "grey", "navy blue", "beige", "khaki", "brown", "white"])


def expected_labels(top, bottom, dupatta):
    required = [UPPER_CLOTHES, PANTS] + ([SCARF] if dupatta else [])
    return {
        "top": UPPER_CLOTHES,
        "bottom": PANTS,
        "dupatta": SCARF if dupatta else None,
        "required": required,
    }


def build_specs(total=1200, seed=1234):
    """A balanced, seeded, shuffled list of `total` generation specs."""
    rng = random.Random(seed)
    specs = []
    for (top, gender, _), count in zip(GROUPS, _group_counts(total)):
        bottoms = _cycle(rng, BOTTOMS[top], count)
        colours = _cycle(rng, COLOURS, count)
        patterns = _cycle(rng, PATTERNS[top], count)
        fabrics = _cycle(rng, FABRICS[top], count)
        backgrounds = _cycle(rng, BACKGROUNDS, count)
        poses = _cycle(rng, POSES, count)
        ages = _cycle(rng, AGES, count)
        people = _cycle(rng, SOUTH_ASIAN_PEOPLE if top in LONG_TOPS else WESTERN_PEOPLE, count)
        dupatta_count = round(count * DUPATTA_SHARE.get(top, 0.0)) if gender == "woman" else 0
        dupattas = [True] * dupatta_count + [False] * (count - dupatta_count)
        rng.shuffle(dupattas)
        for i in range(count):
            bottom = bottoms[i]
            spec = {
                "gender": gender,
                "age": ages[i],
                "people": people[i],
                "top": top,
                "long_top": top in LONG_TOPS,
                "sleeves": "short" if top == "tshirt" else "full",
                "bottom": bottom,
                "dupatta": dupattas[i],
                "colour": colours[i],
                "bottom_colour": _bottom_colour(rng, bottom, colours[i]),
                "dupatta_colour": rng.choice(COLOURS) if dupattas[i] else None,
                "pattern": patterns[i],
                "fabric": fabrics[i],
                "background": backgrounds[i],
                "pose": poses[i],
                "min_hem": MIN_HEM.get(top),
                "max_hem": MAX_HEM.get(top),
            }
            spec["expected"] = expected_labels(top, bottom, spec["dupatta"])
            specs.append(spec)
    rng.shuffle(specs)
    for index, spec in enumerate(specs):
        spec["id"] = f"{index:05d}"
        spec["seed"] = seed * 100000 + index
        spec["split"] = split_for(index)
        spec["prompt"] = positive_prompt(spec)
        spec["negative_prompt"] = NEGATIVE_PROMPT
    return specs


def positive_prompt(spec):
    person = f"{_article(spec['age'])} {spec['age']} {spec['people']} {spec['gender']}"
    garment = _garment_phrase(spec["top"], spec["colour"], spec["pattern"], spec["fabric"])
    parts = [
        f"full body photo of {person} wearing {garment}",
        _bottom_phrase(spec["bottom"], spec["bottom_colour"]),
    ]
    if spec["dupatta"]:
        parts.append(f"{spec['dupatta_colour']} dupatta over the shoulders")
    # CLIP reads at most 77 tokens, so the important words come first.
    parts += [spec["pose"], FRAMING, spec["background"], QUALITY]
    return ", ".join(parts)


def approx_token_count(text):
    """A rough CLIP token count (a word or a punctuation mark each). Real
    BPE splits some rare words further, so keep well under 77."""
    return len(re.findall(r"[A-Za-z]+|[^\sA-Za-z]", text))


def summary(specs):
    """Counts per (top, gender), bottom, dupatta and split, for printing."""
    table = {}
    for key in ("top", "bottom", "dupatta", "split", "gender"):
        counts = {}
        for spec in specs:
            counts[str(spec[key])] = counts.get(str(spec[key]), 0) + 1
        table[key] = dict(sorted(counts.items()))
    return table


In [ ]:
%%writefile /kaggle/working/snapfit_ft/labels.py
"""Automatic labels for the synthetic photos (numpy + OpenCV only).

The current parser (SegFormer-B2 Clothes) labels each generated photo. Its
mistakes on South Asian wear are predictable: the long lower part of a kameez
or kurta comes out as 5 Skirt or 7 Dress, a shalwar is sometimes called a
skirt, and a scarf label (17) appears on garments without a dupatta. Because
every photo was generated from a known spec (prompts.py), those mistakes can
be fixed without a human:

* label 17 is kept only when the spec has a dupatta (otherwise it becomes 4);
* 5 Skirt / 7 Dress pixels above the top's hem that touch the main top
  become 4 Upper-clothes (the whole kameez / kurta is 4);
* 5 / 7 pixels at or below the hem become 6 Pants (shalwar, trousers);
* 5 / 7 pixels above the hem that do not touch the top cannot be decided and
  are ignored in training (255).

Face, hair, arms, legs, shoes, bag and hat are never changed. The hem is the
first row (below the upper body) where pants fill most of the person's width
or the clothes split into two legs.
Then `quality_check` decides whether the photo and its labels are good enough
to train on, and `training_target` blanks out pixels the parser was unsure
about.
"""

import cv2
import numpy as np

NUM_LABELS = 18
IGNORE = 255
LABEL_NAMES = [
    "Background", "Hat", "Hair", "Sunglasses", "Upper-clothes", "Skirt", "Pants",
    "Dress", "Belt", "Left-shoe", "Right-shoe", "Face", "Left-leg", "Right-leg",
    "Left-arm", "Right-arm", "Bag", "Scarf",
]
BACKGROUND, UPPER, SKIRT, PANTS, DRESS, BELT, FACE, SCARF = 0, 4, 5, 6, 7, 8, 11, 17
ARMS = (14, 15)
# Labels the correction may change; everything else is left alone.
MERGE_LABELS = (UPPER, SKIRT, DRESS)
STRAY_LABELS = (SKIRT, DRESS)
# Clothing labels (for the confidence checks and the head check).
GARMENT_LABELS = (UPPER, SKIRT, PANTS, DRESS, BELT, SCARF)

# Hem search: starts this far down the person (above it is the upper body),
# and needs, in each row over a run of HEM_RUN_SHARE of the person's height,
# pants on at least HEM_PANTS_SHARE of the person's pixels or two legs.
HEM_SEARCH_START = 0.35
HEM_PANTS_SHARE = 0.5
HEM_RUN_SHARE = 0.02
# Two-legs test: clothing and bare-leg labels (not arms, so hanging hands do
# not count; not shoes, which are always apart), runs of at least LEG_MIN_RUN
# of the person's width, the second widest at least LEG_WIDTH_RATIO of the
# widest.
LEG_LABELS = (UPPER, SKIRT, PANTS, DRESS, BELT, SCARF, 12, 13)
LEG_MIN_RUN = 0.03
LEG_WIDTH_RATIO = 0.35
# Where the hem probably is when no pants are visible (only reported; such
# photos are dropped by the quality check).
HEM_GUESS = {"kameez": 0.72, "kurta": 0.62}
HEM_GUESS_SHORT = 0.5

# Quality check thresholds (shares of the image or of the person's pixels).
MIN_PERSON_SHARE = 0.06
MAX_PERSON_SHARE = 0.80
MIN_PERSON_PART = 0.015  # of the image: a second blob this big = a second person
MIN_FACE_PART = 0.0002  # of the image
MIN_HEIGHT_SHARE = 0.55  # person height / image height (full body)
MAX_FACE_ROW = 0.25  # face centre within the top quarter of the person
MIN_TOP_SHARE = 0.06
MIN_BOTTOM_SHARE = 0.02
MIN_DUPATTA_SHARE = 0.004
MIN_ARM_SHARE = 0.008  # short sleeves must show some bare arm
MAX_LEFTOVER_SHARE = 0.005  # 5 / 7 left after the correction
MAX_IGNORE_SHARE = 0.15
MIN_GARMENT_CONFIDENCE = 0.75
MIN_GARMENT_CONFIDENCE_RELAXED = 0.60
HEAD_ROWS = 0.08  # top share of the person checked for garment labels
MAX_GARMENT_ON_HEAD = 0.25
HEM_PERCENTILE = 99.5  # lowest row of the top, ignoring specks

# Training target: pixels the parser was unsure about are ignored.
MIN_GARMENT_PROB = 0.5
MIN_OTHER_CONFIDENCE = 0.4

# SAM refinement: only when SAM agrees with the parser's top this well, and
# only background pixels this close to the top become top.
SAM_MIN_IOU = 0.8
SAM_BAND = 8


def person_box(labels):
    """(top_row, bottom_row, left_col, right_col) of all non-background
    pixels, or None when there is no person."""
    person = (labels != BACKGROUND) & (labels != IGNORE)
    if not person.any():
        return None
    rows = np.flatnonzero(person.any(axis=1))
    cols = np.flatnonzero(person.any(axis=0))
    return int(rows[0]), int(rows[-1]), int(cols[0]), int(cols[-1])


def _two_legs(row, min_width):
    """Whether a row of a clothing/leg mask splits into two legs: at least
    two runs of `min_width`+ pixels, the two widest of similar width (a thin
    dupatta end hanging beside a kameez is not a leg)."""
    padded = np.concatenate(([False], row, [False]))
    edges = np.flatnonzero(padded[1:] != padded[:-1])
    widths = np.sort(edges[1::2] - edges[0::2])[::-1]
    widths = widths[widths >= min_width]
    return widths.size >= 2 and widths[1] >= LEG_WIDTH_RATIO * widths[0]


def find_hem(labels, box=None):
    """(row, found): the first row, below HEM_SEARCH_START of the person,
    from which (for a run of rows) either pants fill at least
    HEM_PANTS_SHARE of the person's pixels or the clothes split into two
    legs. A kameez covers the legs down to its hem, so the split also finds
    a shalwar the parser called "skirt". found is False when there is no
    such row."""
    box = box or person_box(labels)
    if box is None:
        return 0, False
    top, bottom, left, right = box
    height = bottom - top + 1
    person = (labels != BACKGROUND) & (labels != IGNORE)
    person_rows = person.sum(axis=1).astype(np.float64)
    pants_rows = (labels == PANTS).sum(axis=1).astype(np.float64)
    share = np.divide(pants_rows, person_rows, out=np.zeros_like(pants_rows), where=person_rows > 0)
    legs = np.isin(labels, LEG_LABELS)
    min_width = max(2, int(round((right - left + 1) * LEG_MIN_RUN)))
    start = top + int(height * HEM_SEARCH_START)
    flags = np.zeros(labels.shape[0], bool)
    for row in range(start, bottom + 1):
        flags[row] = share[row] >= HEM_PANTS_SHARE or _two_legs(legs[row], min_width)
    run = max(1, int(round(height * HEM_RUN_SHARE)))
    for row in range(start, bottom - run + 2):
        if flags[row: row + run].all():
            return row, True
    return bottom + 1, False


def _guess_hem(spec, box):
    top, bottom = box[0], box[1]
    share = HEM_GUESS.get(spec.get("top"), HEM_GUESS_SHORT)
    return top + int((bottom - top + 1) * share)


def correct_labels(labels, spec):
    """(corrected, info): the parser's labels fixed using what the spec says
    the person wears. info has the hem row, whether it was found from the
    pants ("pants") or guessed ("guess"), and how many pixels changed."""
    labels = np.asarray(labels).astype(np.uint8)
    out = labels.copy()
    box = person_box(out)
    info = {"hem_row": None, "hem_frac": None, "hem_source": "none", "changed": 0}
    if box is None:
        return out, info

    if not spec.get("dupatta"):
        out[out == SCARF] = UPPER

    hem, found = find_hem(out, box)
    if not found:
        hem = _guess_hem(spec, box)
    info["hem_row"] = int(hem)
    info["hem_frac"] = (hem - box[0]) / float(box[1] - box[0] + 1)
    info["hem_source"] = "pants" if found else "guess"

    candidates = np.isin(out, MERGE_LABELS).astype(np.uint8)
    count, regions = cv2.connectedComponents(candidates, connectivity=8)
    upper_counts = np.bincount(regions[out == UPPER].ravel(), minlength=count)
    upper_counts[0] = 0
    if upper_counts.max() > 0:
        main = int(upper_counts.argmax())
    else:
        sizes = np.bincount(regions.ravel(), minlength=count)
        sizes[0] = 0
        main = int(sizes.argmax()) if count > 1 else -1
    attached = regions == main if main > 0 else np.zeros(out.shape, bool)

    stray = np.isin(out, STRAY_LABELS)
    below = (np.arange(out.shape[0]) >= hem)[:, None]
    corrected = out.copy()
    corrected[stray & below] = PANTS
    corrected[stray & ~below & attached] = UPPER
    corrected[stray & ~below & ~attached] = IGNORE
    info["changed"] = int((corrected != labels).sum())
    return corrected, info


def _components(mask, min_area):
    count, _, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    keep = [i for i in range(1, count) if stats[i, cv2.CC_STAT_AREA] >= min_area]
    return keep, stats, centroids


def quality_check(corrected, spec, garment_prob=None, info=None, relaxed=False):
    """Whether a corrected label map is good enough to train on.

    Returns {"ok": bool, "reasons": [...], "stats": {...}}. Reasons:
    no_person, person_too_small, person_too_large, several_people, no_face,
    several_faces, not_full_body, face_not_at_top, missing_top, missing_bottom,
    missing_dupatta, top_too_short, top_too_long, hem_unclear,
    skirt_or_dress_left, too_uncertain, low_confidence, garment_on_head,
    no_bare_arms. relaxed=True skips the softer checks (face position, the
    dupatta, bare arms) and lowers the confidence bar.
    """
    corrected = np.asarray(corrected)
    height, width = corrected.shape
    image_area = float(height * width)
    reasons, stats = [], {}

    box = person_box(corrected)
    if box is None:
        return {"ok": False, "reasons": ["no_person"], "stats": stats}
    person = (corrected != BACKGROUND)
    person_area = float(person.sum())
    top, bottom = box[0], box[1]
    person_height = bottom - top + 1
    stats["person_share"] = person_area / image_area
    if stats["person_share"] < MIN_PERSON_SHARE:
        reasons.append("person_too_small")
    if stats["person_share"] > MAX_PERSON_SHARE:
        reasons.append("person_too_large")

    parts, _, _ = _components(person, MIN_PERSON_PART * image_area)
    stats["people"] = len(parts)
    if len(parts) > 1:
        reasons.append("several_people")

    faces, face_stats, face_centres = _components(corrected == FACE, MIN_FACE_PART * image_area)
    stats["faces"] = len(faces)
    if not faces:
        reasons.append("no_face")
    elif len(faces) > 1:
        reasons.append("several_faces")

    stats["height_share"] = person_height / float(height)
    if stats["height_share"] < MIN_HEIGHT_SHARE:
        reasons.append("not_full_body")
    if len(faces) == 1 and not relaxed:
        face_row = (face_centres[faces[0]][1] - top) / float(person_height)
        stats["face_row"] = float(face_row)
        if face_row > MAX_FACE_ROW:
            reasons.append("face_not_at_top")

    def share(ids):
        return float(np.isin(corrected, ids).sum()) / person_area

    stats["top_share"] = share((UPPER,))
    stats["bottom_share"] = share((PANTS,))
    stats["dupatta_share"] = share((SCARF,))
    if stats["top_share"] < MIN_TOP_SHARE:
        reasons.append("missing_top")
    if stats["bottom_share"] < MIN_BOTTOM_SHARE:
        reasons.append("missing_bottom")
    if spec.get("dupatta") and not relaxed and stats["dupatta_share"] < MIN_DUPATTA_SHARE:
        reasons.append("missing_dupatta")

    top_rows = np.nonzero(corrected == UPPER)[0]
    if top_rows.size:
        hem_frac = (np.percentile(top_rows, HEM_PERCENTILE) - top) / float(person_height)
        stats["top_hem_frac"] = float(hem_frac)
        if spec.get("long_top") and spec.get("min_hem") is not None and hem_frac < spec["min_hem"]:
            reasons.append("top_too_short")
        if not spec.get("long_top") and spec.get("max_hem") is not None and hem_frac > spec["max_hem"]:
            reasons.append("top_too_long")

    if info is not None and info.get("hem_source") == "guess":
        reasons.append("hem_unclear")

    stats["leftover_share"] = share(STRAY_LABELS)
    if stats["leftover_share"] > MAX_LEFTOVER_SHARE:
        reasons.append("skirt_or_dress_left")
    stats["ignore_share"] = share((IGNORE,))
    if stats["ignore_share"] > MAX_IGNORE_SHARE:
        reasons.append("too_uncertain")

    garment = np.isin(corrected, GARMENT_LABELS)
    if garment_prob is not None and garment.any():
        stats["garment_confidence"] = float(np.asarray(garment_prob, dtype=np.float64)[garment].mean())
        bar = MIN_GARMENT_CONFIDENCE_RELAXED if relaxed else MIN_GARMENT_CONFIDENCE
        if stats["garment_confidence"] < bar:
            reasons.append("low_confidence")

    head = slice(top, top + max(1, int(person_height * HEAD_ROWS)))
    head_person = float(person[head].sum())
    stats["garment_on_head"] = float(garment[head].sum()) / head_person if head_person else 0.0
    if stats["garment_on_head"] > MAX_GARMENT_ON_HEAD:
        reasons.append("garment_on_head")

    stats["arm_share"] = share(ARMS)
    if spec.get("sleeves") == "short" and not relaxed and stats["arm_share"] < MIN_ARM_SHARE:
        reasons.append("no_bare_arms")

    return {"ok": not reasons, "reasons": reasons, "stats": stats}


def training_target(corrected, confidence, garment_prob):
    """The label map to train on: `corrected` with unsure pixels set to
    IGNORE. confidence: the parser's top probability per pixel; garment_prob:
    its summed probability of the clothing labels (both 0..1)."""
    target = np.asarray(corrected).astype(np.uint8).copy()
    confidence = np.asarray(confidence, dtype=np.float32)
    garment_prob = np.asarray(garment_prob, dtype=np.float32)
    garment = np.isin(target, GARMENT_LABELS)
    other = ~garment & (target != IGNORE)
    target[garment & (garment_prob < MIN_GARMENT_PROB)] = IGNORE
    target[other & (confidence < MIN_OTHER_CONFIDENCE)] = IGNORE
    return target


def top_box(corrected):
    """(x0, y0, x1, y1) of the top garment (label 4), or None."""
    ys, xs = np.nonzero(np.asarray(corrected) == UPPER)
    if not ys.size:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def refine_with_sam(corrected, sam_mask, min_iou=SAM_MIN_IOU, band=SAM_BAND):
    """(labels, used): grow the top (4) into background pixels that SAM puts
    on the garment, within `band` pixels of the parser's top. Used only when
    SAM's mask overlaps the parser's top by at least `min_iou` (otherwise SAM
    found something else and the labels are returned unchanged). Never
    touches face, hair, arms, legs or any other label."""
    corrected = np.asarray(corrected)
    top = corrected == UPPER
    sam = np.asarray(sam_mask).astype(bool)
    if sam.shape != top.shape or not top.any() or not sam.any():
        return corrected, False
    iou = (top & sam).sum() / float((top | sam).sum())
    if iou < min_iou:
        return corrected, False
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * band + 1, 2 * band + 1))
    near = cv2.dilate(top.astype(np.uint8), kernel) > 0
    refined = corrected.copy()
    refined[sam & near & (corrected == BACKGROUND)] = UPPER
    return refined, True


In [ ]:
%%writefile /kaggle/working/snapfit_ft/metrics.py
"""Scores for the clothes parser (numpy only).

* mIoU: for each label, intersection / union of predicted and true pixels
  over the whole evaluation set, averaged over the labels that appear in the
  truth or the prediction (a label that appears in neither is skipped).
  Pixels whose true label is the ignore index (255) are not counted.
* kameez consistency: share of kameez / kurta photos where at least 90% of
  the garment is labelled 4 Upper-clothes and less than 5% 5 Skirt / 7 Dress.
  That is what SnapFit needs: its try-on mask covers labels 4 and 7, and a
  "skirt" under a long top is what leaves old cloth behind.
* gate: whether the fine-tuned parser should replace the base one.
"""

import math

import numpy as np

NUM_LABELS = 18
IGNORE = 255
UPPER, SKIRT, DRESS = 4, 5, 7

MIN_TOP_SHARE = 0.90
MAX_SKIRT_DRESS_SHARE = 0.05
# Western mIoU may drop by at most this many points (mIoU in percent).
MAX_WESTERN_DROP = 1.0


def confusion_matrix(pred, truth, num_classes=NUM_LABELS, ignore_index=IGNORE):
    """num_classes x num_classes counts: rows = true label, columns =
    predicted label. Pixels with the ignore index (or any out-of-range id) in
    the truth are skipped; out-of-range predictions are skipped too."""
    pred = np.asarray(pred).astype(np.int64).ravel()
    truth = np.asarray(truth).astype(np.int64).ravel()
    if pred.shape != truth.shape:
        raise ValueError(f"shape mismatch: pred {pred.shape} vs truth {truth.shape}")
    valid = (truth != ignore_index) & (truth >= 0) & (truth < num_classes) & (pred >= 0) & (pred < num_classes)
    index = truth[valid] * num_classes + pred[valid]
    return np.bincount(index, minlength=num_classes * num_classes).reshape(num_classes, num_classes)


def per_class_iou(matrix):
    """IoU per label (0..1), NaN for labels absent from both truth and
    prediction."""
    matrix = np.asarray(matrix, dtype=np.float64)
    intersection = np.diag(matrix)
    union = matrix.sum(axis=0) + matrix.sum(axis=1) - intersection
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(union > 0, intersection / np.where(union > 0, union, 1), np.nan)


def mean_iou(matrix):
    """Mean IoU in percent over the labels that appear (NaN if none)."""
    iou = per_class_iou(matrix)
    if np.all(np.isnan(iou)):
        return float("nan")
    return float(np.nanmean(iou) * 100.0)


def pixel_accuracy(matrix):
    matrix = np.asarray(matrix, dtype=np.float64)
    total = matrix.sum()
    return float(np.trace(matrix) / total * 100.0) if total else float("nan")


def garment_shares(pred, garment_mask):
    """(share labelled 4, share labelled 5 or 7) of the garment pixels."""
    pred = np.asarray(pred)
    garment = np.asarray(garment_mask).astype(bool)
    total = garment.sum()
    if not total:
        return float("nan"), float("nan")
    values = pred[garment]
    return float((values == UPPER).sum() / total), float(np.isin(values, (SKIRT, DRESS)).sum() / total)


def kameez_consistent(pred, garment_mask, min_top=MIN_TOP_SHARE, max_skirt=MAX_SKIRT_DRESS_SHARE):
    """Whether the garment (a kameez / kurta) is labelled as one top: at
    least `min_top` of it 4 and less than `max_skirt` of it 5 / 7."""
    top, skirt = garment_shares(pred, garment_mask)
    if math.isnan(top):
        return False
    return top >= min_top and skirt < max_skirt


def rate(flags):
    """Share of True values, in percent (NaN for an empty list)."""
    flags = list(flags)
    return float(sum(bool(f) for f in flags) / len(flags) * 100.0) if flags else float("nan")


def _finite(*values):
    for value in values:
        try:
            number = float(value)
        except (TypeError, ValueError):
            return False
        if not math.isfinite(number):
            return False
    return True


def gate(western_base, western_finetuned, kameez_base, kameez_finetuned, max_drop=MAX_WESTERN_DROP):
    """The rule for using the fine-tuned parser: Western mIoU drops by at
    most `max_drop` points AND kameez consistency improves (strictly).
    Returns {"recommended", "western_drop", "kameez_gain", "reasons"}."""
    reasons = []
    if not _finite(western_base, western_finetuned, kameez_base, kameez_finetuned):
        return {"recommended": False, "western_drop": None, "kameez_gain": None, "reasons": ["missing or invalid scores"]}
    drop = western_base - western_finetuned
    gain = kameez_finetuned - kameez_base
    if drop > max_drop:
        reasons.append(f"Western mIoU dropped {drop:.2f} points (allowed: {max_drop:.2f})")
    if gain <= 0:
        reasons.append(f"kameez consistency did not improve ({kameez_base:.1f}% -> {kameez_finetuned:.1f}%)")
    return {"recommended": not reasons, "western_drop": drop, "kameez_gain": gain, "reasons": reasons}


def choose_best(history, base_western, max_drop=MAX_WESTERN_DROP):
    """Index of the best epoch in `history` (dicts with "western_miou",
    "kameez", "synthetic_miou", all on validation sets). Among the epochs
    whose Western mIoU is within `max_drop` of the base, the one with the
    best kameez consistency (then synthetic mIoU, then Western mIoU); if no
    epoch is within the limit, the one with the best Western mIoU. None for
    an empty history."""
    if not history:
        return None

    def number(entry, key):
        value = entry.get(key)
        return value if _finite(value) else -1e9

    allowed = [i for i, e in enumerate(history) if _finite(e.get("western_miou"), base_western) and base_western - e["western_miou"] <= max_drop]
    if allowed:
        return max(allowed, key=lambda i: (number(history[i], "kameez"), number(history[i], "synthetic_miou"), number(history[i], "western_miou"), -i))
    return max(range(len(history)), key=lambda i: (number(history[i], "western_miou"), -i))


def iou_table(names, base_iou, finetuned_iou):
    """Markdown rows: label | base IoU | fine-tuned IoU | change (percent)."""
    def fmt(value):
        return "–" if value is None or not math.isfinite(value) else f"{value * 100:.1f}"

    rows = ["| Label | Base | Fine-tuned | Change |", "|---|---|---|---|"]
    for name, base, tuned in zip(names, base_iou, finetuned_iou):
        if (base is None or not math.isfinite(base)) and (tuned is None or not math.isfinite(tuned)):
            continue
        change = (tuned - base) * 100 if _finite(base, tuned) else None
        change_text = "–" if change is None else f"{change:+.1f}"
        rows.append(f"| {name} | {fmt(base)} | {fmt(tuned)} | {change_text} |")
    return "\n".join(rows)


In [ ]:
%%writefile /kaggle/working/snapfit_ft/augment.py
"""Training augmentation for the clothes parser (numpy + OpenCV only).

At run time the parser's processor squeezes the whole photo to 512x512, so
training does the same: a random crop that keeps most of the photo (or the
whole photo), squeezed to 512x512, a horizontal flip that also swaps the
left / right labels, and a small colour jitter. Labels are resized with
nearest-neighbour so no new ids appear; the ignore value (255) survives.
"""

import cv2
import numpy as np

IGNORE = 255
# Left / right label pairs (ATR): shoes, legs, arms.
FLIP_PAIRS = ((9, 10), (12, 13), (14, 15))

SIZE = 512
FULL_IMAGE_CHANCE = 0.3
SCALE = (0.6, 1.0)  # share of the photo's area kept by a crop
ASPECT_JITTER = (0.8, 1.25)  # crop aspect / photo aspect
BRIGHTNESS = 0.25
CONTRAST = 0.25
SATURATION = 0.25
HUE_DEGREES = 8


def flip_labels(labels):
    """A label map mirrored left-right, with left / right ids swapped."""
    flipped = np.ascontiguousarray(np.asarray(labels)[:, ::-1])
    swapped = flipped.copy()
    for left, right in FLIP_PAIRS:
        swapped[flipped == left] = right
        swapped[flipped == right] = left
    return swapped


def random_crop_box(height, width, rng, scale=SCALE, aspect_jitter=ASPECT_JITTER, full_chance=FULL_IMAGE_CHANCE):
    """(y0, x0, h, w) of a random crop inside a height x width photo."""
    if rng.random() < full_chance:
        return 0, 0, height, width
    for _ in range(10):
        area = height * width * rng.uniform(*scale)
        aspect = (width / float(height)) * float(np.exp(rng.uniform(np.log(aspect_jitter[0]), np.log(aspect_jitter[1]))))
        w = int(round(np.sqrt(area * aspect)))
        h = int(round(np.sqrt(area / aspect)))
        if 0 < w <= width and 0 < h <= height:
            y0 = int(rng.integers(0, height - h + 1))
            x0 = int(rng.integers(0, width - w + 1))
            return y0, x0, h, w
    return 0, 0, height, width


def color_jitter(image, rng, brightness=BRIGHTNESS, contrast=CONTRAST, saturation=SATURATION, hue_degrees=HUE_DEGREES):
    """uint8 RGB image with random brightness, contrast, saturation and hue."""
    out = image.astype(np.float32)
    out *= rng.uniform(1 - brightness, 1 + brightness)
    mean = out.mean()
    out = (out - mean) * rng.uniform(1 - contrast, 1 + contrast) + mean
    grey = out.mean(axis=2, keepdims=True)
    out = (out - grey) * rng.uniform(1 - saturation, 1 + saturation) + grey
    out = np.clip(out, 0, 255).astype(np.uint8)
    if hue_degrees:
        hsv = cv2.cvtColor(out, cv2.COLOR_RGB2HSV)
        # OpenCV hue runs 0..179 (2 degrees per step).
        shift = int(round(rng.uniform(-hue_degrees, hue_degrees) / 2.0))
        hsv[..., 0] = ((hsv[..., 0].astype(np.int16) + shift) % 180).astype(np.uint8)
        out = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return out


def train_transform(image, labels, rng, size=SIZE):
    """(image, labels) ready for training: uint8 RGB size x size and uint8
    labels size x size. `labels` must have the image's height and width."""
    image = np.asarray(image)
    labels = np.asarray(labels).astype(np.uint8)
    if image.shape[:2] != labels.shape:
        raise ValueError(f"image {image.shape[:2]} and labels {labels.shape} differ in size")
    y0, x0, h, w = random_crop_box(image.shape[0], image.shape[1], rng)
    image = image[y0: y0 + h, x0: x0 + w]
    labels = labels[y0: y0 + h, x0: x0 + w]
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    labels = cv2.resize(labels, (size, size), interpolation=cv2.INTER_NEAREST)
    if rng.random() < 0.5:
        image = np.ascontiguousarray(image[:, ::-1])
        labels = flip_labels(labels)
    image = color_jitter(image, rng)
    return image, labels


def eval_transform(image, size=SIZE):
    """uint8 RGB size x size (the whole photo squeezed, as the processor does)."""
    return cv2.resize(np.asarray(image), (size, size), interpolation=cv2.INTER_LINEAR)


def normalize(image, mean, std):
    """float32 CHW array: (image / 255 - mean) / std."""
    out = image.astype(np.float32) / 255.0
    out = (out - np.asarray(mean, np.float32)) / np.asarray(std, np.float32)
    return np.ascontiguousarray(out.transpose(2, 0, 1))


In [ ]:
%%writefile /kaggle/working/snapfit_ft/viz.py
"""Picture grids for the report (numpy + OpenCV only): photo | base parser |
fine-tuned parser, one row per example, with a colour legend."""

import cv2
import numpy as np

LABEL_NAMES = [
    "Background", "Hat", "Hair", "Sunglasses", "Upper-clothes", "Skirt", "Pants",
    "Dress", "Belt", "Left-shoe", "Right-shoe", "Face", "Left-leg", "Right-leg",
    "Left-arm", "Right-arm", "Bag", "Scarf",
]
# RGB colours, one per ATR label; the ones that matter most for SnapFit
# (4 top, 5 skirt, 6 pants, 7 dress, 17 scarf) are strong and distinct.
PALETTE = np.array([
    [0, 0, 0],        # 0 Background
    [128, 64, 0],     # 1 Hat
    [255, 200, 0],    # 2 Hair
    [90, 90, 90],     # 3 Sunglasses
    [220, 20, 60],    # 4 Upper-clothes (red)
    [255, 140, 0],    # 5 Skirt (orange)
    [30, 90, 255],    # 6 Pants (blue)
    [190, 0, 255],    # 7 Dress (purple)
    [120, 120, 0],    # 8 Belt
    [0, 120, 120],    # 9 Left-shoe
    [0, 170, 170],    # 10 Right-shoe
    [255, 220, 180],  # 11 Face
    [160, 255, 160],  # 12 Left-leg
    [100, 200, 100],  # 13 Right-leg
    [255, 180, 220],  # 14 Left-arm
    [220, 130, 180],  # 15 Right-arm
    [140, 100, 60],   # 16 Bag
    [0, 230, 120],    # 17 Scarf (green)
], dtype=np.uint8)
IGNORE_COLOUR = np.array([255, 255, 255], dtype=np.uint8)


def colorize(labels):
    """RGB image of a label map (255 = ignore shown white)."""
    labels = np.asarray(labels)
    out = np.empty(labels.shape + (3,), np.uint8)
    out[:] = IGNORE_COLOUR
    valid = labels < len(PALETTE)
    out[valid] = PALETTE[labels[valid]]
    return out


def _fit(item, height):
    """A photo (H x W x 3) resized smoothly, or a label map (H x W) resized
    with nearest-neighbour and coloured, to the given height."""
    item = np.asarray(item)
    width = max(1, int(round(item.shape[1] * height / float(item.shape[0]))))
    if item.ndim == 2:
        return colorize(cv2.resize(item.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST))
    return cv2.resize(item.astype(np.uint8), (width, height), interpolation=cv2.INTER_AREA)


def _text(canvas, text, x, y, scale=0.5, colour=(0, 0, 0)):
    cv2.putText(canvas, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, scale, colour, 1, cv2.LINE_AA)


def legend(width, labels=range(len(LABEL_NAMES))):
    """White strip with a colour box and the name of each label."""
    labels = list(labels)
    per_row = max(1, width // 150)
    rows = (len(labels) + per_row - 1) // per_row
    canvas = np.full((rows * 24 + 8, width, 3), 255, np.uint8)
    for n, label in enumerate(labels):
        x = 8 + (n % per_row) * 150
        y = 8 + (n // per_row) * 24
        canvas[y: y + 16, x: x + 16] = PALETTE[label]
        cv2.rectangle(canvas, (x, y), (x + 15, y + 15), (0, 0, 0), 1)
        _text(canvas, f"{label} {LABEL_NAMES[label]}", x + 22, y + 13, 0.42)
    return canvas


def comparison_grid(rows, titles=("photo", "base parser", "fine-tuned"), row_height=320, captions=None):
    """One RGB image: each row is (photo, base labels, fine-tuned labels);
    label maps are coloured. A header, optional row captions and a legend
    are added."""
    if not rows:
        raise ValueError("no rows")
    rendered = []
    for n, row in enumerate(rows):
        cells = []
        for item in row:
            cell = _fit(item, row_height)
            cells.append(np.pad(cell, ((0, 0), (4, 4), (0, 0)), constant_values=255))
        line = np.concatenate(cells, axis=1)
        if captions:
            bar = np.full((22, line.shape[1], 3), 255, np.uint8)
            _text(bar, str(captions[n])[:120], 6, 16, 0.45)
            line = np.concatenate([bar, line], axis=0)
        rendered.append(line)
    width = max(r.shape[1] for r in rendered)
    rendered = [np.pad(r, ((0, 4), (0, width - r.shape[1]), (0, 0)), constant_values=255) for r in rendered]
    header = np.full((30, width, 3), 255, np.uint8)
    column = width // max(1, len(rows[0]))
    for n, title in enumerate(titles):
        _text(header, title, n * column + 10, 21, 0.6)
    return np.concatenate([header] + rendered + [legend(width)], axis=0)


In [ ]:
%%writefile /kaggle/working/snapfit_ft/gen_worker.py
"""Stable Diffusion 1.5 generation worker: one process per GPU.

Started by the notebook with CUDA_VISIBLE_DEVICES set to one GPU. Takes every
`--num-workers`-th spec starting at `--worker`, skips specs already done
(a PNG, or a "flagged" marker when the safety checker blocked the image),
and stops at `--deadline` (a Unix time). Each image is written to a temporary
file first and renamed, so a stopped run never leaves half a PNG behind.
"""

import argparse
import json
import os
import sys
import time
import traceback


def _log(*parts):
    print(time.strftime("%H:%M:%S"), *parts, flush=True)


def _done(out_dir, spec_id):
    return os.path.exists(os.path.join(out_dir, "images", spec_id + ".png")) or os.path.exists(
        os.path.join(out_dir, "flagged", spec_id + ".txt")
    )


def _load_pipeline(model, variant):
    import torch
    from diffusers import DPMSolverMultistepScheduler, StableDiffusionPipeline

    last_error = None
    for attempt in ([variant, None] if variant else [None]):
        try:
            kwargs = {"torch_dtype": torch.float16}
            if attempt:
                kwargs["variant"] = attempt
            pipe = StableDiffusionPipeline.from_pretrained(model, **kwargs)
            break
        except Exception as error:  # a missing fp16 variant falls back to the full weights
            last_error = error
            _log(f"loading with variant={attempt!r} failed: {error!r}")
    else:
        raise last_error
    if getattr(pipe, "safety_checker", None) is None:
        _log("WARNING: this pipeline has no safety checker; flagged images cannot be dropped")
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.set_progress_bar_config(disable=True)
    return pipe.to("cuda")


def _save(out_dir, spec, image):
    path = os.path.join(out_dir, "images", spec["id"] + ".png")
    image.save(path + ".tmp", format="PNG")
    os.replace(path + ".tmp", path)
    spec_path = os.path.join(out_dir, "specs", spec["id"] + ".json")
    with open(spec_path + ".tmp", "w", encoding="utf-8") as handle:
        json.dump(spec, handle, indent=1)
    os.replace(spec_path + ".tmp", spec_path)


def _flag(out_dir, spec):
    with open(os.path.join(out_dir, "flagged", spec["id"] + ".txt"), "w", encoding="utf-8") as handle:
        handle.write("blocked by the safety checker\n")


def _generate(pipe, batch, args):
    import torch

    return pipe(
        prompt=[spec["prompt"] for spec in batch],
        negative_prompt=[spec["negative_prompt"] for spec in batch],
        width=args.width,
        height=args.height,
        num_inference_steps=args.steps,
        guidance_scale=args.guidance,
        generator=[torch.Generator(device="cuda").manual_seed(int(spec["seed"])) for spec in batch],
    )


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--specs", required=True)
    parser.add_argument("--out", required=True)
    parser.add_argument("--worker", type=int, default=0)
    parser.add_argument("--num-workers", type=int, default=1)
    parser.add_argument("--deadline", type=float, default=0.0)
    parser.add_argument("--model", default="stable-diffusion-v1-5/stable-diffusion-v1-5")
    parser.add_argument("--variant", default="fp16")
    parser.add_argument("--steps", type=int, default=22)
    parser.add_argument("--guidance", type=float, default=7.0)
    parser.add_argument("--width", type=int, default=512)
    parser.add_argument("--height", type=int, default=768)
    parser.add_argument("--batch", type=int, default=2)
    args = parser.parse_args(argv)

    import torch

    for sub in ("images", "specs", "flagged"):
        os.makedirs(os.path.join(args.out, sub), exist_ok=True)
    with open(args.specs, encoding="utf-8") as handle:
        specs = json.load(handle)
    mine = specs[args.worker:: args.num_workers]
    todo = [spec for spec in mine if not _done(args.out, spec["id"])]
    _log(f"worker {args.worker}: {len(mine)} specs, {len(todo)} to do, GPU {torch.cuda.get_device_name(0)}")
    if not todo:
        return 0

    pipe = _load_pipeline(args.model, args.variant)
    saved = flagged = failed = 0
    start = time.time()
    batch_size = max(1, args.batch)
    position = 0
    while position < len(todo):
        if args.deadline and time.time() > args.deadline:
            _log(f"worker {args.worker}: time budget reached, stopping")
            break
        batch = todo[position: position + batch_size]
        try:
            result = _generate(pipe, batch, args)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if batch_size > 1:
                batch_size = 1
                _log(f"worker {args.worker}: out of GPU memory, batch size -> 1")
                continue
            _log(f"worker {args.worker}: out of GPU memory on {batch[0]['id']}, skipping it")
            failed += 1
            position += 1
            continue
        except Exception:
            _log(f"worker {args.worker}: generation failed for {[s['id'] for s in batch]}:\n{traceback.format_exc()}")
            failed += len(batch)
            position += len(batch)
            if failed > 20 and saved == 0:
                _log("too many failures and nothing saved; giving up")
                return 1
            continue
        nsfw = getattr(result, "nsfw_content_detected", None) or [False] * len(batch)
        for spec, image, blocked in zip(batch, result.images, nsfw):
            if blocked:
                _flag(args.out, spec)
                flagged += 1
            else:
                _save(args.out, spec, image)
                saved += 1
        position += len(batch)
        done = saved + flagged
        if done and (done % 10 < len(batch) or position >= len(todo)):
            seconds = (time.time() - start) / done
            left = (len(todo) - position) * seconds / 60.0
            _log(f"worker {args.worker}: {position}/{len(todo)} done ({saved} saved, {flagged} flagged, {failed} failed), {seconds:.1f} s/image, about {left:.0f} min left")
    _log(f"worker {args.worker}: finished: {saved} saved, {flagged} flagged by the safety checker, {failed} failed")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
import importlib

if WORK not in sys.path:
    sys.path.insert(0, WORK)
import snapfit_ft.augment as ftaugment
import snapfit_ft.labels as ftlabels
import snapfit_ft.metrics as ftmetrics
import snapfit_ft.prompts as prompts
import snapfit_ft.viz as ftviz

for module in (prompts, ftlabels, ftmetrics, ftaugment, ftviz):
    importlib.reload(module)
print("Helper modules loaded.")

## Stage 1 – Generation specs (a few seconds)

In [ ]:
started = stage("Stage 1/7 - generation specs")
specs = prompts.build_specs(NUM_IMAGES, SPEC_SEED)
SPEC_BY_ID = {spec["id"]: spec for spec in specs}
SPECS_PATH = os.path.join(FT, "specs.json")
with open(SPECS_PATH, "w", encoding="utf-8") as handle:
    json.dump(specs, handle)
for key, counts in prompts.summary(specs).items():
    print(f"{key:8s}", counts)
print("Longest prompt (rough CLIP tokens):", max(prompts.approx_token_count(s["prompt"]) for s in specs))
for spec in specs[:3]:
    print("-", spec["id"], spec["split"], "|", spec["prompt"])
TIMINGS["specs_min"] = (time.time() - started) / 60

## Stage 2 – Generate the photos (about 80 minutes)
One Stable Diffusion 1.5 process per GPU. The safety checker is on; blocked
images are dropped. Already generated photos are skipped, so the cell can be run
again after an interruption. Logs: `/kaggle/working/ft_work/logs/gen_gpu*.log`.

In [ ]:
started = stage("Stage 2/7 - generating photos with Stable Diffusion 1.5")
GEN_DONE = os.path.join(FT, "generation_done.json")


def count_generated():
    images = len(glob.glob(os.path.join(SYNTH, "images", "*.png")))
    flagged = len(glob.glob(os.path.join(SYNTH, "flagged", "*.txt")))
    return images, flagged


def log_tail(path, lines=15):
    try:
        with open(path, encoding="utf-8", errors="replace") as handle:
            return "".join(handle.readlines()[-lines:])
    except OSError:
        return "(no log)"


if os.path.exists(GEN_DONE):
    print("Already done:", open(GEN_DONE).read())
else:
    variant = "fp16"
    try:
        from diffusers import StableDiffusionPipeline

        print("Downloading Stable Diffusion 1.5 once (both workers then load it from the cache) ...")
        try:
            StableDiffusionPipeline.download(SD_MODEL, variant="fp16")
        except Exception as error:
            print("fp16 files not available, downloading the full weights:", repr(error)[:200])
            variant = ""
            StableDiffusionPipeline.download(SD_MODEL)
    except Exception as error:
        print("Pre-download did not work; each worker downloads by itself:", repr(error)[:300])

    deadline = time.time() + GEN_TIME_BUDGET_MIN * 60
    workers = max(1, N_GPUS)
    processes = []
    try:
        for worker in range(workers):
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(worker), PYTHONUNBUFFERED="1")
            log_path = os.path.join(LOGS, f"gen_gpu{worker}.log")
            command = [
                sys.executable, os.path.join(LIB, "gen_worker.py"),
                "--specs", SPECS_PATH, "--out", SYNTH,
                "--worker", str(worker), "--num-workers", str(workers),
                "--deadline", str(deadline), "--model", SD_MODEL, "--variant", variant,
                "--steps", str(GEN_STEPS), "--guidance", str(GEN_GUIDANCE),
                "--width", str(GEN_WIDTH), "--height", str(GEN_HEIGHT), "--batch", str(GEN_BATCH),
            ]
            log_file = open(log_path, "a", encoding="utf-8")
            processes.append((subprocess.Popen(command, env=env, stdout=log_file, stderr=subprocess.STDOUT), log_file, log_path))
            print(f"Started worker {worker} on GPU {worker}; log: {log_path}")

        first = sum(count_generated())
        watch_start = time.time()
        last_tail = time.time()
        while any(process.poll() is None for process, _, _ in processes):
            time.sleep(60)
            images, flagged = count_generated()
            new = images + flagged - first
            minutes = (time.time() - watch_start) / 60
            per_min = new / minutes if minutes > 0 else 0.0
            left = (len(specs) - images - flagged) / per_min if per_min > 0 else float("nan")
            budget_left = max(0.0, (deadline - time.time()) / 60)
            print(f"[{elapsed_min():6.1f} min] {images} saved + {flagged} blocked of {len(specs)} | "
                  f"{per_min:.1f} photos/min | about {min(left, budget_left):.0f} min left "
                  f"(budget {budget_left:.0f} min)", flush=True)
            if time.time() - last_tail > 600:
                last_tail = time.time()
                for _, _, log_path in processes:
                    print("   ", log_tail(log_path, 1).strip())
    finally:
        for process, log_file, _ in processes:
            if process.poll() is None:
                process.terminate()
            log_file.close()

    for worker, (process, _, log_path) in enumerate(processes):
        code = process.wait()
        print(f"Worker {worker} exit code {code}. Last log lines:\n{log_tail(log_path, 4)}")
        if code != 0:
            print(f"Worker {worker} failed. Log tail:\n{log_tail(log_path, 40)}")
    images, flagged = count_generated()
    if images == 0:
        raise RuntimeError("No photos were generated. Send the log lines printed above to the lead.")
    with open(GEN_DONE, "w") as handle:
        json.dump({"images": images, "blocked_by_safety_checker": flagged, "minutes": (time.time() - started) / 60}, handle)

GENERATED, BLOCKED = count_generated()
print(f"Photos: {GENERATED} saved, {BLOCKED} blocked by the safety checker.")
TIMINGS["generation_min"] = (time.time() - started) / 60

## Stage 3 – Automatic labels (about 10 minutes)
The current parser labels each photo. The labels are then corrected using the
spec (`labels.py`): the whole kameez / kurta becomes 4, a shalwar 6, a dupatta 17.
Photos with unclear labels are dropped; the reasons are counted below.

In [ ]:
started = stage("Stage 3/7 - automatic labels")
import numpy as np
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModelForSemanticSegmentation, SegformerImageProcessor

DEVICE = "cuda:0"
LABEL_LOG = os.path.join(FT, "label_log.jsonl")
LABEL_DONE = os.path.join(FT, "labels_done.json")
processor = SegformerImageProcessor.from_pretrained(BASE_PARSER)
GARMENT_IDS = list(ftlabels.GARMENT_LABELS)


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB")


def parse_with_confidence(model, images):
    """(labels, top probability, clothing probability) per PIL image, at the
    image's own size, exactly as the try-on server parses (processor at 512,
    logits upsampled, argmax)."""
    inputs = processor(images=images, return_tensors="pt")
    with torch.inference_mode():
        logits = model(pixel_values=inputs["pixel_values"].to(DEVICE)).logits
        results = []
        for i, image in enumerate(images):
            up = F.interpolate(logits[i: i + 1].float(), size=(image.height, image.width), mode="bilinear", align_corners=False)
            probs = up.softmax(dim=1)[0]
            confidence, labels = probs.max(dim=0)
            garment = probs[GARMENT_IDS].sum(dim=0)
            results.append((labels.to(torch.uint8).cpu().numpy(), confidence.cpu().numpy(), garment.cpu().numpy()))
    return results


def read_label_log():
    records = {}
    if os.path.exists(LABEL_LOG):
        with open(LABEL_LOG, encoding="utf-8") as handle:
            for line in handle:
                try:
                    record = json.loads(line)
                    records[record["id"]] = record
                except (ValueError, KeyError):
                    pass  # a line cut off by an interruption
    return records


records = read_label_log()
image_ids = sorted(os.path.basename(p)[:-4] for p in glob.glob(os.path.join(SYNTH, "images", "*.png")))
todo = [i for i in image_ids if i not in records]
print(f"{len(image_ids)} photos, {len(records)} already labelled, {len(todo)} to label")

if todo:
    base_model = AutoModelForSemanticSegmentation.from_pretrained(BASE_PARSER).to(DEVICE).eval()
    sam_model = sam_processor = None
    if USE_SAM:
        try:
            from transformers import SamModel, SamProcessor

            sam_processor = SamProcessor.from_pretrained(SAM_MODEL)
            sam_model = SamModel.from_pretrained(SAM_MODEL).to(DEVICE).eval()
            print("SAM loaded for edge refinement.")
        except Exception as error:
            print("SAM not available, labels are used without it:", repr(error)[:200])
            sam_model = None

    def sam_top_mask(image, box):
        inputs = sam_processor(image, input_boxes=[[list(box)]], return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            outputs = sam_model(**inputs, multimask_output=True)
        masks = sam_processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), inputs["reshaped_input_sizes"].cpu())
        best = int(outputs.iou_scores[0, 0].argmax())
        return masks[0][0, best].numpy().astype(bool)

    sam_errors = 0
    with open(LABEL_LOG, "a", encoding="utf-8") as log:
        for start in range(0, len(todo), 16):
            chunk = todo[start: start + 16]
            pictures = [load_rgb(os.path.join(SYNTH, "images", i + ".png")) for i in chunk]
            for spec_id, picture, (raw, confidence, garment_prob) in zip(chunk, pictures, parse_with_confidence(base_model, pictures)):
                spec = SPEC_BY_ID.get(spec_id)
                if spec is None:
                    continue
                corrected, info = ftlabels.correct_labels(raw, spec)
                sam_used = False
                if sam_model is not None:
                    try:
                        box = ftlabels.top_box(corrected)
                        if box is not None:
                            corrected, sam_used = ftlabels.refine_with_sam(corrected, sam_top_mask(picture, box))
                    except Exception as error:
                        sam_errors += 1
                        if sam_errors <= 3:
                            print("SAM skipped for", spec_id, repr(error)[:200])
                strict = ftlabels.quality_check(corrected, spec, garment_prob, info)
                relaxed = ftlabels.quality_check(corrected, spec, garment_prob, info, relaxed=True)
                target = ftlabels.training_target(corrected, confidence, garment_prob)
                Image.fromarray(raw).save(os.path.join(SYNTH, "parsed", spec_id + ".png"))
                Image.fromarray(target).save(os.path.join(SYNTH, "target", spec_id + ".png"))
                record = {
                    "id": spec_id, "split": spec["split"], "top": spec["top"], "long_top": spec["long_top"],
                    "dupatta": spec["dupatta"], "ok_strict": strict["ok"], "reasons_strict": strict["reasons"],
                    "ok_relaxed": relaxed["ok"], "reasons_relaxed": relaxed["reasons"],
                    "hem_source": info["hem_source"], "changed_pixels": info["changed"], "sam_used": sam_used,
                    "stats": strict["stats"],
                }
                log.write(json.dumps(record) + "\n")
                records[spec_id] = record
            log.flush()
            if (start // 16) % 10 == 0:
                print(f"[{elapsed_min():6.1f} min] labelled {min(start + 16, len(todo))}/{len(todo)}", flush=True)
    del base_model, sam_model
    torch.cuda.empty_cache()

all_records = [records[i] for i in image_ids if i in records]
strict_rate = sum(r["ok_strict"] for r in all_records) / max(1, len(all_records))
USE_RELAXED = strict_rate < 0.5
OK_KEY = "ok_relaxed" if USE_RELAXED else "ok_strict"
if USE_RELAXED:
    print(f"Strict check kept only {strict_rate:.0%}; using the relaxed check instead.")
kept = [r for r in all_records if r[OK_KEY]]
SYNTH_TRAIN = [r["id"] for r in kept if r["split"] == "train"]
SYNTH_VAL = [r["id"] for r in kept if r["split"] == "val"]
SYNTH_TEST = [r["id"] for r in kept if r["split"] == "test"]

reason_counts = {}
for record in all_records:
    for reason in record["reasons_relaxed" if USE_RELAXED else "reasons_strict"]:
        reason_counts[reason] = reason_counts.get(reason, 0) + 1
by_top = {}
for record in all_records:
    entry = by_top.setdefault(record["top"], [0, 0])
    entry[0] += 1
    entry[1] += bool(record[OK_KEY])
print(f"Kept {len(kept)} of {len(all_records)} photos ({len(kept) / max(1, len(all_records)):.0%}): "
      f"train {len(SYNTH_TRAIN)}, val {len(SYNTH_VAL)}, test {len(SYNTH_TEST)}")
for top, (total, ok) in sorted(by_top.items()):
    print(f"  {top:7s} kept {ok}/{total}")
print("Drop reasons (a photo can have several):")
for reason, count in sorted(reason_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {reason:22s} {count}")
print("SAM refinement used on", sum(r["sam_used"] for r in all_records), "photos")
if len(kept) < 0.6 * max(1, len(all_records)):
    print("NOTE: fewer than 60% kept. Training still runs; see the drop reasons above.")
LABEL_SUMMARY = {
    "photos": len(all_records), "kept": len(kept), "strict_keep_rate": strict_rate, "relaxed_check_used": USE_RELAXED,
    "train": len(SYNTH_TRAIN), "val": len(SYNTH_VAL), "test": len(SYNTH_TEST),
    "kept_by_top": {top: {"total": t, "kept": k} for top, (t, k) in by_top.items()},
    "drop_reasons": reason_counts,
}
with open(LABEL_DONE, "w") as handle:
    json.dump(LABEL_SUMMARY, handle, indent=1)
TIMINGS["labels_min"] = (time.time() - started) / 60

## Stage 4 – Western data (about 5 minutes)
The ATR clothes dataset (the one the base parser was trained on). A fixed set of
500 photos is held out for testing and never used in training.

In [ ]:
started = stage("Stage 4/7 - Western data (ATR)")
from datasets import load_dataset

western = None
for attempt in range(3):
    try:
        western = load_dataset(WESTERN_DATASET, split="train")
        break
    except Exception as error:
        print(f"Loading the dataset failed (try {attempt + 1}/3):", repr(error)[:300])
        time.sleep(20)
if western is None:
    raise RuntimeError("Could not load the ATR dataset. Check that Internet is On, then run this cell again.")
print(western)
columns = western.column_names
IMAGE_COLUMN = "image" if "image" in columns else columns[0]
MASK_COLUMN = "mask" if "mask" in columns else [c for c in columns if c != IMAGE_COLUMN][0]
print("Using columns:", IMAGE_COLUMN, "(photo) and", MASK_COLUMN, "(labels)")


def western_item(index):
    """(PIL RGB photo, uint8 label map at the photo's size)."""
    row = western[int(index)]
    image = row[IMAGE_COLUMN].convert("RGB")
    mask = np.array(row[MASK_COLUMN])
    if mask.ndim == 3:
        mask = mask[..., 0]
    mask = mask.astype(np.uint8)
    if mask.shape != (image.height, image.width):
        mask = np.array(Image.fromarray(mask).resize((image.width, image.height), Image.NEAREST))
    mask[(mask >= ftlabels.NUM_LABELS) & (mask != ftlabels.IGNORE)] = ftlabels.IGNORE
    return image, mask


sample_ids = sorted({int(v) for index in range(5) for v in np.unique(western_item(index)[1])})
print("Label ids in the first photos:", sample_ids)
if max(sample_ids) >= ftlabels.NUM_LABELS and max(sample_ids) != ftlabels.IGNORE:
    print("WARNING: unexpected label ids in the dataset")

order = np.random.default_rng(WESTERN_SPLIT_SEED).permutation(len(western))
WESTERN_TEST = [int(i) for i in order[:WESTERN_TEST_SIZE]]
WESTERN_VAL = [int(i) for i in order[WESTERN_TEST_SIZE: WESTERN_TEST_SIZE + WESTERN_VAL_SIZE]]
pool = order[WESTERN_TEST_SIZE + WESTERN_VAL_SIZE:]
n_western_train = int(min(MAX_WESTERN_TRAIN, max(MIN_WESTERN_TRAIN, len(SYNTH_TRAIN))))
WESTERN_TRAIN = [int(i) for i in pool[:n_western_train]]
with open(os.path.join(FT, "western_split.json"), "w") as handle:
    json.dump({"seed": WESTERN_SPLIT_SEED, "test": WESTERN_TEST, "val": WESTERN_VAL, "train": WESTERN_TRAIN}, handle)
print(f"Western: test {len(WESTERN_TEST)}, val {len(WESTERN_VAL)}, train {len(WESTERN_TRAIN)} "
      f"(synthetic train {len(SYNTH_TRAIN)})")
TIMINGS["western_min"] = (time.time() - started) / 60

## Shared helpers for training and evaluation

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

IMAGE_MEAN = processor.image_mean
IMAGE_STD = processor.image_std
BEST_DIR = os.path.join(CKPT, "best")
LAST_CKPT = os.path.join(CKPT, "last.pt")
TRAIN_DONE = os.path.join(CKPT, "train_done.json")


class MixDataset(Dataset):
    """Synthetic photos with their corrected labels + ATR photos."""

    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        source, key = self.items[index]
        if source == "synth":
            image = np.array(load_rgb(os.path.join(SYNTH, "images", key + ".png")))
            with Image.open(os.path.join(SYNTH, "target", key + ".png")) as target:
                labels = np.array(target)
        else:
            picture, labels = western_item(key)
            image = np.array(picture)
        rng = np.random.default_rng([torch.initial_seed() % 2**32, index])
        image, labels = ftaugment.train_transform(image, labels, rng)
        x = torch.from_numpy(ftaugment.normalize(image, IMAGE_MEAN, IMAGE_STD))
        y = torch.from_numpy(labels.astype(np.int64))
        return x, y


class LogitsOnly(nn.Module):
    """Returns plain logits (so DataParallel can gather them); autocast is
    inside forward because it is per thread."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        with torch.autocast("cuda", dtype=torch.float16):
            return self.model(pixel_values=pixel_values).logits


def predict_labels(model, images, batch_size=16):
    """Label map per PIL image at its own size, as the try-on server does."""
    model.eval()
    out = []
    for start in range(0, len(images), batch_size):
        chunk = images[start: start + batch_size]
        inputs = processor(images=chunk, return_tensors="pt")
        with torch.inference_mode():
            logits = model(pixel_values=inputs["pixel_values"].to(DEVICE)).logits
            for i, image in enumerate(chunk):
                up = F.interpolate(logits[i: i + 1].float(), size=(image.height, image.width), mode="bilinear", align_corners=False)
                out.append(up.argmax(dim=1)[0].to(torch.uint8).cpu().numpy())
    return out


def evaluate(model, western_ids, synth_ids, keep_western=(), keep_synth=(), chunk=32):
    """Scores on Western photos (ATR labels) and synthetic photos (automatic
    labels). Returns a dict; predictions for the keep_* ids are included
    for the picture grids."""
    keep_western, keep_synth = set(keep_western), set(keep_synth)
    western_cm = np.zeros((ftmetrics.NUM_LABELS, ftmetrics.NUM_LABELS), np.int64)
    synth_cm = np.zeros_like(western_cm)
    kameez_flags, kept = [], {}
    for start in range(0, len(western_ids), chunk):
        ids = western_ids[start: start + chunk]
        pairs = [western_item(i) for i in ids]
        for i, (picture, truth), pred in zip(ids, pairs, predict_labels(model, [p for p, _ in pairs])):
            western_cm += ftmetrics.confusion_matrix(pred, truth)
            if i in keep_western:
                kept[("western", i)] = pred
    for start in range(0, len(synth_ids), chunk):
        ids = synth_ids[start: start + chunk]
        pictures = [load_rgb(os.path.join(SYNTH, "images", i + ".png")) for i in ids]
        for i, pred in zip(ids, predict_labels(model, pictures)):
            with Image.open(os.path.join(SYNTH, "target", i + ".png")) as target:
                truth = np.array(target)
            synth_cm += ftmetrics.confusion_matrix(pred, truth)
            if SPEC_BY_ID[i]["long_top"]:
                kameez_flags.append(ftmetrics.kameez_consistent(pred, truth == ftlabels.UPPER))
            if i in keep_synth:
                kept[("synth", i)] = pred
    return {
        "western_miou": ftmetrics.mean_iou(western_cm),
        "western_iou": [None if math.isnan(v) else float(v) for v in ftmetrics.per_class_iou(western_cm)],
        "western_pixel_accuracy": ftmetrics.pixel_accuracy(western_cm),
        "synthetic_miou": ftmetrics.mean_iou(synth_cm),
        "synthetic_iou": [None if math.isnan(v) else float(v) for v in ftmetrics.per_class_iou(synth_cm)],
        "kameez": ftmetrics.rate(kameez_flags),
        "kameez_photos": len(kameez_flags),
        "predictions": kept,
    }


print("Helpers ready.")

## Stage 5 – Fine-tune (about 75 minutes)
Starts from the base parser (same 18 labels). Each epoch is saved, so the cell
can be run again after an interruption and continues where it stopped. The best
epoch is picked on the validation sets (never on the test sets).

In [ ]:
started = stage("Stage 5/7 - fine-tuning")
TRAINED = False
TRAIN_SUMMARY = {}
if os.path.exists(TRAIN_DONE):
    TRAIN_SUMMARY = json.load(open(TRAIN_DONE))
    TRAINED = os.path.isdir(BEST_DIR)
    print("Already trained:", {k: v for k, v in TRAIN_SUMMARY.items() if k != "history"})
elif len(SYNTH_TRAIN) < MIN_SYNTH_TRAIN:
    TRAIN_SUMMARY = {"skipped": f"only {len(SYNTH_TRAIN)} synthetic training photos (need {MIN_SYNTH_TRAIN})"}
    print("Training skipped:", TRAIN_SUMMARY["skipped"])
else:
    torch.manual_seed(0)
    random.seed(0)
    model = AutoModelForSemanticSegmentation.from_pretrained(BASE_PARSER).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    try:
        scaler = torch.amp.GradScaler("cuda")
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler()
    state = {"epoch": 0, "step": 0, "history": [], "planned_epochs": MAX_EPOCHS, "base_val": None,
             "train_seconds": 0.0, "batch_per_gpu": BATCH_PER_GPU, "best_index": None}
    if os.path.exists(LAST_CKPT):
        checkpoint = torch.load(LAST_CKPT, map_location="cpu", weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        state = checkpoint["state"]
        print(f"Resuming after epoch {state['epoch']} (step {state['step']}).")

    net = LogitsOnly(model)
    if N_GPUS > 1:
        net = nn.DataParallel(net)

    def validation_scores():
        scores = evaluate(model, WESTERN_VAL, SYNTH_VAL)
        model.train()
        return {k: scores[k] for k in ("western_miou", "synthetic_miou", "kameez", "kameez_photos")}

    if state["base_val"] is None:
        state["base_val"] = validation_scores()
        print("Base parser on the validation sets:", state["base_val"])

    # Find a batch size that fits (halve on out-of-memory).
    batch_per_gpu = state["batch_per_gpu"]
    while True:
        try:
            model.train()
            dummy = torch.randn(batch_per_gpu * max(1, N_GPUS), 3, 512, 512, device=DEVICE)
            dummy_logits = net(dummy)
            dummy_logits.float().mean().backward()
            optimizer.zero_grad(set_to_none=True)
            del dummy, dummy_logits
            torch.cuda.empty_cache()
            break
        except torch.cuda.OutOfMemoryError:
            optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            if batch_per_gpu <= 2:
                raise
            batch_per_gpu //= 2
            print("Out of GPU memory; batch per GPU ->", batch_per_gpu)
    state["batch_per_gpu"] = batch_per_gpu

    items = [("synth", i) for i in SYNTH_TRAIN] + [("western", i) for i in WESTERN_TRAIN]
    loader = DataLoader(MixDataset(items), batch_size=batch_per_gpu * max(1, N_GPUS), shuffle=True,
                        num_workers=min(4, os.cpu_count() or 2), pin_memory=True, drop_last=True,
                        persistent_workers=True)
    steps_per_epoch = len(loader)
    budget_seconds = max(10 * 60, min(TRAIN_TIME_BUDGET_MIN * 60, (minutes_left() - 30) * 60))
    print(f"{len(items)} training photos ({len(SYNTH_TRAIN)} synthetic + {len(WESTERN_TRAIN)} Western), "
          f"batch {batch_per_gpu * max(1, N_GPUS)}, {steps_per_epoch} steps per epoch, "
          f"time budget {budget_seconds / 60:.0f} min")

    def learning_rate(step):
        total = max(1, state["planned_epochs"] * steps_per_epoch)
        warm = min(1.0, (step + 1) / float(WARMUP_STEPS))
        return LEARNING_RATE * warm * max(0.0, 1.0 - step / float(total)) ** POLY_POWER

    out_of_time = False
    for epoch in range(state["epoch"] + 1, MAX_EPOCHS + 1):
        if epoch > state["planned_epochs"] or out_of_time:
            break
        model.train()
        epoch_start = time.time()
        losses = []
        for x, y in loader:
            if (y != ftlabels.IGNORE).sum() == 0:
                continue
            for group in optimizer.param_groups:
                group["lr"] = learning_rate(state["step"])
            logits = net(x.to(DEVICE, non_blocking=True))
            y = y.to(DEVICE, non_blocking=True)
            logits = F.interpolate(logits.float(), size=y.shape[-2:], mode="bilinear", align_corners=False)
            loss = F.cross_entropy(logits, y, ignore_index=ftlabels.IGNORE)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            state["step"] += 1
            if math.isfinite(loss.item()):
                losses.append(loss.item())
            if state["step"] % 50 == 0:
                print(f"[{elapsed_min():6.1f} min] epoch {epoch} step {state['step']} "
                      f"loss {np.mean(losses[-50:]):.4f} lr {learning_rate(state['step']):.2e}", flush=True)
            if state["train_seconds"] + (time.time() - epoch_start) > budget_seconds:
                out_of_time = True
                print("Training time budget reached; finishing this epoch early.")
                break
        scores = validation_scores()
        epoch_seconds = time.time() - epoch_start
        state["train_seconds"] += epoch_seconds
        state["epoch"] = epoch
        state["history"].append({"epoch": epoch, "loss": float(np.mean(losses)) if losses else None,
                                 "seconds": epoch_seconds, **scores})
        if epoch == 1 or len(state["history"]) == 1:
            fit = int(budget_seconds // max(1.0, epoch_seconds))
            state["planned_epochs"] = int(min(MAX_EPOCHS, max(1, fit)))
            print(f"One epoch takes {epoch_seconds / 60:.1f} min -> planning {state['planned_epochs']} epochs")
        best = ftmetrics.choose_best(state["history"], state["base_val"]["western_miou"], MAX_WESTERN_DROP)
        if best == len(state["history"]) - 1:
            model.save_pretrained(BEST_DIR)
            processor.save_pretrained(BEST_DIR)
            state["best_index"] = best
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "state": state}, LAST_CKPT + ".tmp")
        os.replace(LAST_CKPT + ".tmp", LAST_CKPT)
        print(f"[{elapsed_min():6.1f} min] epoch {epoch} done in {epoch_seconds / 60:.1f} min | "
              f"Western val mIoU {scores['western_miou']:.2f} (base {state['base_val']['western_miou']:.2f}) | "
              f"kameez val {scores['kameez']:.1f}% (base {state['base_val']['kameez']:.1f}%) | "
              f"best epoch so far: {state['history'][state['best_index']]['epoch'] if state['best_index'] is not None else '-'}",
              flush=True)
    TRAINED = os.path.isdir(BEST_DIR)
    TRAIN_SUMMARY = {"epochs": state["epoch"], "steps": state["step"], "minutes": state["train_seconds"] / 60,
                     "batch": state["batch_per_gpu"] * max(1, N_GPUS), "base_val": state["base_val"],
                     "best_epoch": state["history"][state["best_index"]]["epoch"] if state["best_index"] is not None else None,
                     "history": state["history"], "train_photos": {"synthetic": len(SYNTH_TRAIN), "western": len(WESTERN_TRAIN)}}
    with open(TRAIN_DONE, "w") as handle:
        json.dump(TRAIN_SUMMARY, handle, indent=1)
    del loader, net, model, optimizer
    torch.cuda.empty_cache()
TIMINGS["training_min"] = (time.time() - started) / 60

## Stage 6 – Evaluate base vs fine-tuned (about 5–10 minutes)
Western: the 500 held-out ATR photos. South Asian: the held-out synthetic photos
(never trained on), scored against the **automatic** labels.

In [ ]:
started = stage("Stage 6/7 - evaluation")
GRID_SYNTH = []
for wanted in (("kameez", True), ("kameez", False), ("kurta", True), ("kurta", False)):
    GRID_SYNTH += [i for i in SYNTH_TEST if (SPEC_BY_ID[i]["top"], SPEC_BY_ID[i]["dupatta"]) == wanted][:3]
GRID_SYNTH = (GRID_SYNTH + [i for i in SYNTH_TEST if SPEC_BY_ID[i]["long_top"] and i not in GRID_SYNTH])[:12]
GRID_WESTERN = WESTERN_TEST[:6]

base_model = AutoModelForSemanticSegmentation.from_pretrained(BASE_PARSER).to(DEVICE).eval()
RESULTS = {"base": evaluate(base_model, WESTERN_TEST, SYNTH_TEST, GRID_WESTERN, GRID_SYNTH)}
del base_model
print(f"Base:       Western mIoU {RESULTS['base']['western_miou']:.2f} | kameez {RESULTS['base']['kameez']:.1f}% "
      f"of {RESULTS['base']['kameez_photos']} | synthetic mIoU {RESULTS['base']['synthetic_miou']:.2f}")
if TRAINED:
    tuned_model = AutoModelForSemanticSegmentation.from_pretrained(BEST_DIR).to(DEVICE).eval()
    RESULTS["finetuned"] = evaluate(tuned_model, WESTERN_TEST, SYNTH_TEST, GRID_WESTERN, GRID_SYNTH)
    print(f"Fine-tuned: Western mIoU {RESULTS['finetuned']['western_miou']:.2f} | kameez {RESULTS['finetuned']['kameez']:.1f}% "
          f"| synthetic mIoU {RESULTS['finetuned']['synthetic_miou']:.2f}")
else:
    tuned_model = None
    print("No fine-tuned model to evaluate.")

for kind, ids in (("south_asian", GRID_SYNTH), ("western", GRID_WESTERN)):
    try:
        rows, captions = [], []
        for i in ids:
            key = ("synth", i) if kind == "south_asian" else ("western", i)
            if key not in RESULTS["base"]["predictions"]:
                continue
            if kind == "south_asian":
                photo = np.array(load_rgb(os.path.join(SYNTH, "images", i + ".png")))
                spec = SPEC_BY_ID[i]
                captions.append(f"{i}: {spec['top']} + {spec['bottom']}" + (" + dupatta" if spec["dupatta"] else ""))
            else:
                photo = np.array(western_item(i)[0])
                captions.append(f"ATR test photo {i}")
            base_pred = RESULTS["base"]["predictions"][key]
            tuned_pred = RESULTS["finetuned"]["predictions"][key] if TRAINED else np.full_like(base_pred, ftlabels.IGNORE)
            rows.append((photo, base_pred, tuned_pred))
        if rows:
            grid = ftviz.comparison_grid(rows, captions=captions)
            Image.fromarray(grid).save(os.path.join(REPORT, f"grid_{kind}.png"))
            print("Saved", os.path.join(REPORT, f"grid_{kind}.png"))
    except Exception:
        print(f"Could not draw the {kind} grid (the scores are not affected):\n{traceback.format_exc()}")
TIMINGS["evaluation_min"] = (time.time() - started) / 60

## Stage 7 – Gate, save and report
The fine-tuned parser is **recommended** only if the Western mIoU drops by at most
1.0 point **and** the kameez consistency improves.

In [ ]:
started = stage("Stage 7/7 - gate, save and report")
base, tuned = RESULTS["base"], RESULTS.get("finetuned")
if tuned is not None:
    GATE = ftmetrics.gate(base["western_miou"], tuned["western_miou"], base["kameez"], tuned["kameez"], MAX_WESTERN_DROP)
else:
    GATE = {"recommended": False, "western_drop": None, "kameez_gain": None,
            "reasons": ["no fine-tuned model (" + TRAIN_SUMMARY.get("skipped", "training did not finish") + ")"]}
RECOMMENDED = bool(GATE["recommended"])


def number(value, digits=2):
    return None if value is None or not math.isfinite(value) else round(float(value), digits)


METRICS = {
    "recommended": RECOMMENDED,
    "western_miou_base": number(base["western_miou"]),
    "western_miou_finetuned": number(tuned["western_miou"]) if tuned else None,
    "kameez_consistency_base": number(base["kameez"]),
    "kameez_consistency_finetuned": number(tuned["kameez"]) if tuned else None,
    "synthetic_miou_base": number(base["synthetic_miou"]),
    "synthetic_miou_finetuned": number(tuned["synthetic_miou"]) if tuned else None,
    "units": "mIoU and kameez consistency are percentages (0-100)",
    "gate": {"rule": f"Western mIoU drop <= {MAX_WESTERN_DROP} points AND kameez consistency improves",
             "western_drop": number(GATE["western_drop"]), "kameez_gain": number(GATE["kameez_gain"]),
             "reasons_not_recommended": GATE["reasons"]},
    "test_sets": {"western_atr_photos": len(WESTERN_TEST), "synthetic_photos": len(SYNTH_TEST),
                  "synthetic_kameez_kurta_photos": base["kameez_photos"],
                  "synthetic_labels": "automatic (base parser + spec-based correction), not hand-drawn"},
    "per_class_iou_western": {"labels": ftlabels.LABEL_NAMES, "base": base["western_iou"],
                              "finetuned": tuned["western_iou"] if tuned else None},
    "per_class_iou_synthetic": {"labels": ftlabels.LABEL_NAMES, "base": base["synthetic_iou"],
                                "finetuned": tuned["synthetic_iou"] if tuned else None},
    "data": {"generated": GENERATED, "blocked_by_safety_checker": BLOCKED, "labels": LABEL_SUMMARY,
             "western_split_seed": WESTERN_SPLIT_SEED},
    "training": {k: v for k, v in TRAIN_SUMMARY.items()},
    "timings_min": {k: round(v, 1) for k, v in TIMINGS.items()},
    "models": {"base_parser": BASE_PARSER, "generator": SD_MODEL, "sam": SAM_MODEL if USE_SAM else None,
               "western_data": WESTERN_DATASET},
}

if tuned_model is not None:
    tuned_model.save_pretrained(MODEL_OUT)
    processor.save_pretrained(MODEL_OUT)
    with open(os.path.join(MODEL_OUT, "snapfit_finetune.json"), "w") as handle:
        json.dump({k: METRICS[k] for k in ("recommended", "western_miou_base", "western_miou_finetuned",
                                           "kameez_consistency_base", "kameez_consistency_finetuned")}, handle, indent=1)
    print("Model saved to", MODEL_OUT, "(recommended)" if RECOMMENDED else "(NOT recommended)")
def json_safe(value):
    """NaN / infinity -> None (strict JSON), recursively."""
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(value, (np.floating, np.integer, np.bool_)):
        return json_safe(value.item())
    return value


METRICS = json_safe(METRICS)
with open(os.path.join(REPORT, "metrics.json"), "w") as handle:
    json.dump(METRICS, handle, indent=1, allow_nan=False)


def show(value, suffix=""):
    return "n/a" if value is None else f"{value:.1f}{suffix}"


lines = [
    "# SnapFit clothes parser fine-tuning – results",
    "",
    f"**Verdict: {'USE IT' if RECOMMENDED else 'DO NOT USE IT, keep the base model'}**",
    "",
    "| Test | Base parser | Fine-tuned | Change |",
    "|---|---|---|---|",
]
for title, key in (("Western photos: mIoU (ATR, %d held-out photos)" % len(WESTERN_TEST), "western_miou"),
                   ("Kameez / kurta labelled as one top (%d synthetic test photos)" % base["kameez_photos"], "kameez_consistency"),
                   ("Synthetic photos: mIoU against the automatic labels", "synthetic_miou")):
    b, t = METRICS[key + "_base"], METRICS[key + "_finetuned"]
    unit = "%" if key == "kameez_consistency" else ""
    change = None if b is None or t is None else f"{t - b:+.1f}"
    lines.append(f"| {title} | {show(b, unit)} | {show(t, unit)} | {change or 'n/a'} |")
lines += [
    "",
    f"**Rule:** the fine-tuned parser is used only if the Western mIoU drops by at most {MAX_WESTERN_DROP} point "
    "and the kameez result gets better.",
]
if GATE["reasons"]:
    lines += ["", "Why it is not recommended:"] + [f"- {r}" for r in GATE["reasons"]]
lines += [
    "",
    "## What the numbers mean",
    "- **mIoU** (mean intersection over union): for each label, how much the predicted area and the true area "
    "overlap, averaged over the labels. 100 = perfect.",
    "- **Kameez / kurta labelled as one top**: share of kameez and kurta photos where at least 90% of the garment "
    "is labelled *Upper-clothes* and less than 5% *Skirt* or *Dress*. SnapFit needs this to remove the old top fully.",
    "- The synthetic test photos were generated by Stable Diffusion and labelled **automatically** (the base parser "
    "plus fixes based on what each photo was asked to show). Scores against them show the direction of the change, "
    "not an exact accuracy. Only photos that passed the label quality check are in the test set.",
    "- The base parser was trained on ATR, so it may have seen some of the Western test photos before. The Western "
    "score mainly shows whether fine-tuning made it forget anything.",
    "",
    "## Data",
    f"- Generated photos: {GENERATED} (and {BLOCKED} blocked by the safety checker).",
    f"- Kept after the label check: {LABEL_SUMMARY['kept']} ({LABEL_SUMMARY['kept'] / max(1, LABEL_SUMMARY['photos']):.0%})"
    f"{' using the relaxed check' if LABEL_SUMMARY['relaxed_check_used'] else ''}: "
    f"train {LABEL_SUMMARY['train']}, validation {LABEL_SUMMARY['val']}, test {LABEL_SUMMARY['test']}.",
    f"- Western (ATR) photos: train {len(WESTERN_TRAIN)}, validation {len(WESTERN_VAL)}, test {len(WESTERN_TEST)} (seed {WESTERN_SPLIT_SEED}).",
    "- Most common drop reasons: " + ", ".join(f"{k} ({v})" for k, v in sorted(LABEL_SUMMARY["drop_reasons"].items(), key=lambda kv: -kv[1])[:6]),
    "",
    "## Training",
]
if TRAIN_SUMMARY.get("history"):
    lines.append(f"- {TRAIN_SUMMARY['epochs']} epochs in {TRAIN_SUMMARY['minutes']:.0f} min, batch {TRAIN_SUMMARY['batch']}; "
                 f"best epoch (picked on the validation sets): {TRAIN_SUMMARY['best_epoch']}.")
    lines += ["", "| Epoch | Loss | Western val mIoU | Kameez val | Synthetic val mIoU |", "|---|---|---|---|---|"]
    for row in TRAIN_SUMMARY["history"]:
        lines.append(f"| {row['epoch']} | {show(row['loss'])} | {show(row['western_miou'])} | {show(row['kameez'], '%')} | {show(row['synthetic_miou'])} |")
else:
    lines.append("- " + TRAIN_SUMMARY.get("skipped", "No training was done."))
lines += ["", "## Western per-label IoU (%)", "",
          ftmetrics.iou_table(ftlabels.LABEL_NAMES, base["western_iou"], tuned["western_iou"] if tuned else [None] * 18),
          "", "## Pictures", "- `grid_south_asian.png` and `grid_western.png`: photo | base parser | fine-tuned. "
          "Red = Upper-clothes, orange = Skirt, blue = Pants, purple = Dress, green = Scarf.",
          "", "## Licences",
          "- Base parser `mattmdjaga/segformer_b2_clothes`: MIT. The fine-tuned model is a derivative of it.",
          "- Photo generator `stable-diffusion-v1-5/stable-diffusion-v1-5`: CreativeML OpenRAIL-M (its use restrictions apply "
          "to the generated photos).",
          "- Edge refinement `facebook/sam-vit-base`: Apache-2.0.",
          "- Western data `mattmdjaga/human_parsing_dataset` (ATR): see its dataset card; used for research / evaluation.",
          "", f"Total run time: {elapsed_min():.0f} min."]
with open(os.path.join(REPORT, "summary.md"), "w", encoding="utf-8") as handle:
    handle.write("\n".join(lines) + "\n")
TIMINGS["report_min"] = (time.time() - started) / 60
print("\n".join(lines[:12]))

## Tidy up the output
Kaggle keeps everything in `/kaggle/working` as this notebook's output. The many
small work files are packed into one archive (`synthetic_data.tar`) and the work
folder is deleted, so the output stays small and easy to attach.

In [ ]:
stage("Tidying up the output")
try:
    os.makedirs(os.path.join(REPORT, "logs"), exist_ok=True)
    for path in glob.glob(os.path.join(LOGS, "*.log")) + [LABEL_LOG, os.path.join(FT, "western_split.json"), TRAIN_DONE]:
        if os.path.exists(path):
            shutil.copy(path, os.path.join(REPORT, "logs", os.path.basename(path)))
    if CLEANUP_AT_END:
        if KEEP_SYNTH_ARCHIVE:
            archive = os.path.join(WORK, "synthetic_data.tar")
            with tarfile.open(archive + ".tmp", "w") as tar:
                for sub in ("images", "specs", "target"):
                    tar.add(os.path.join(SYNTH, sub), arcname=sub)
                tar.add(SPECS_PATH, arcname="specs.json")
            os.replace(archive + ".tmp", archive)
            print(f"Synthetic photos and labels packed into {archive} ({os.path.getsize(archive) / 2**30:.2f} GB)")
        shutil.rmtree(FT, ignore_errors=True)
        print("Work folder removed.")
except Exception:
    print("Tidying up failed (the results are not affected):\n" + traceback.format_exc())

## Verdict and next steps

In [ ]:
print("=" * 70)
if RECOMMENDED:
    print("VERDICT: USE IT - the fine-tuned parser is better on kameez/kurta and keeps Western quality.")
else:
    print("VERDICT: DO NOT USE IT, keep the base model.")
    for reason in GATE["reasons"]:
        print("  -", reason)
print(f"Western mIoU: base {show(METRICS['western_miou_base'])} -> fine-tuned {show(METRICS['western_miou_finetuned'])}")
print(f"Kameez/kurta as one top: base {show(METRICS['kameez_consistency_base'], '%')} -> "
      f"fine-tuned {show(METRICS['kameez_consistency_finetuned'], '%')}")
print("=" * 70)
print("Next steps:")
print(" 1. If you ran this interactively: Save Version (the Save & Run All commit keeps the output).")
print(" 2. In the try-on server notebook: Add Input -> Your Work -> this notebook.")
print("    The server uses /kaggle/input/<this notebook>/segformer-snapfit/ only when")
print("    report/metrics.json says \"recommended\": true; otherwise it keeps the base parser.")
print(" 3. Send report/summary.md and the two grid pictures to the lead.")
print(f"Output: {REPORT} and {MODEL_OUT if os.path.isdir(MODEL_OUT) else '(no model folder)'}")